# CoOp Benchmark Dataset Downloader

This notebook downloads the CoOp benchmark datasets following the **exact** procedure from:
https://github.com/KaiyangZhou/CoOp/blob/main/DATASETS.md

**Excluded** (as per project spec): ImageNetV2, ImageNet-Sketch, ImageNet-A, ImageNet-R

**Included datasets and their folder names under `$DATA`:**

| | | |
|---|---|---|
| `caltech-101/` | `oxford_pets/` | `stanford_cars/` |
| `oxford_flowers/` | `food-101/` | `fgvc_aircraft/` |
| `sun397/` | `dtd/` | `eurosat/` |
| `ucf101/` | `imagenet/` (optional) | |

## Step 1 — Install Dependencies

In [ ]:
# torch & torchvision are pre-installed on Colab
!pip install -q gdown datasets

## Step 2 — Configuration

Set `SELECTED_DATASETS` to download only specific datasets, or leave as `None` to download all.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────

# DATA_ROOT is set in the Google Drive cell below — no need to change it here.

# Set to a list of dataset names to download only those, or None for all.
# Available: caltech-101, oxford_pets, stanford_cars, oxford_flowers,
#            food-101, fgvc_aircraft, sun397, dtd, eurosat, ucf101
SELECTED_DATASETS = None  # e.g. ["dtd", "eurosat", "ucf101"]

# ImageNet options (skip by default — too large for Colab)
INCLUDE_IMAGENET = False

# ─────────────────────────────────────────────────────────────────────

### Mount Google Drive & Link Shared Folder

The datasets will be saved to a **shared Google Drive folder** you have edit access to.

**One-time setup (do this once in your browser):**
1. Open this link while logged into your **Colab Google account**:
   https://drive.google.com/drive/folders/1KS7WnD_rnE351gXuyKJeoyBNF1Lrnfrm
2. Right-click the folder name at the top → **"Organize"** → **"Add shortcut"** → select **My Drive** → click **"Add"**

That creates a shortcut in your Drive so Colab can see it after mounting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# List your Drive root to find the shortcut name
import os
print("Contents of My Drive root:")
for item in sorted(os.listdir("/content/drive/MyDrive")):
    print(f"  📁 {item}" if os.path.isdir(f"/content/drive/MyDrive/{item}") else f"  📄 {item}")

In [ ]:
# ── Shared folder path (shortcut added to My Drive) ──────────────────
DATA_ROOT = "/content/drive/MyDrive/AML Dataset"

# Verify the folder is accessible
assert os.path.isdir(DATA_ROOT), (
    f"❌ Folder not found: {DATA_ROOT}\n"
    f"   Make sure you added the shortcut to My Drive (see instructions above)."
)
print(f"✅ DATA_ROOT = {DATA_ROOT}")
print(f"   Contents: {os.listdir(DATA_ROOT)[:15]}")

## Step 3 — Imports

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import gdown
import torchvision.datasets as tvd

## Step 4 — Download Functions

Uses `torchvision.datasets` for 7 datasets (with folder renaming where needed).
Falls back to Kaggle (Stanford Cars), HuggingFace (SUN397), and Google Drive (UCF101) for the rest.
Also downloads CoOp split JSON files via `gdown`.

In [ ]:
# ── CoOp split JSON files (hosted on Google Drive) ───────────────────
SPLIT_FILES = {
    "caltech-101":    [("split_zhou_Caltech101.json", "1hyarUivQE36mY6jSomru6Fjd-JzwcCzN")],
    "oxford_pets":    [("split_zhou_OxfordPets.json", "1501r8Ber4nNKvmlFVQZ8SeUHTcdTTEqs")],
    "stanford_cars":  [("split_zhou_StanfordCars.json", "1ObCFbaAgVu0I-k_Au-gIUcefirdAuizT")],
    "oxford_flowers": [
        ("cat_to_name.json", "1AkcxCXeK_RCGCEC_GvmWxjcjaNhu-at0"),
        ("split_zhou_OxfordFlowers.json", "1Pp0sRXzZFZq15zVOzKjKBu4A9i01nozT"),
    ],
    "food-101":       [("split_zhou_Food101.json", "1QK0tGi096I0Ba6kggatX1ee6dJFIcEJl")],
    "fgvc_aircraft":  [],
    "sun397":         [("split_zhou_SUN397.json", "1y2RD81BYuiyvebdN-JymPfyWYcd8_MUq")],
    "dtd":            [("split_zhou_DescribableTextures.json", "1u3_QfB467jqHgNXC00UIzbLZRQCg2S7x")],
    "eurosat":        [("split_zhou_EuroSAT.json", "1Ip7yaCWFi0eaOFUGga0lUdVi_DDQth1o")],
    "ucf101":         [("split_zhou_UCF101.json", "1I0S0q91hJfsV9Gf4xDIjgDq4AqBNJb1y")],
}


def _download_splits(ds_name: str, ds_root: Path):
    """Download CoOp split JSON files from Google Drive."""
    ds_root.mkdir(parents=True, exist_ok=True)
    for fname, gid in SPLIT_FILES.get(ds_name, []):
        dest = ds_root / fname
        if dest.exists():
            print(f"    [skip] {fname}")
            continue
        print(f"    [gdown] {fname}")
        gdown.download(id=gid, output=str(dest), quiet=True)


# ── torchvision-based downloads ──────────────────────────────────────

def dl_caltech101(data_root: Path):
    """caltech-101/101_ObjectCategories/ — torchvision.Caltech101"""
    ds_root = data_root / "caltech-101"
    if (ds_root / "101_ObjectCategories").exists():
        print("  ✓ Already exists")
    else:
        ds_root.mkdir(parents=True, exist_ok=True)
        tvd.Caltech101(root=str(data_root), download=True)
        # Fix: torchvision creates 'caltech101', CoOp needs 'caltech-101'
        tv_dir = data_root / "caltech101"
        if tv_dir.exists():
            for item in tv_dir.iterdir():
                dest = ds_root / item.name
                if not dest.exists():
                    shutil.move(str(item), str(dest))
            shutil.rmtree(str(tv_dir), ignore_errors=True)

        # Fix potential double nesting (zip may contain caltech-101/ folder)
        nested = ds_root / "caltech-101" / "101_ObjectCategories"
        if nested.exists() and not (ds_root / "101_ObjectCategories").exists():
            for item in (ds_root / "caltech-101").iterdir():
                dest = ds_root / item.name
                if not dest.exists():
                  shutil.move(str(item), str(ds_root / item.name))
            shutil.rmtree(str(ds_root / "caltech-101"), ignore_errors=True)
        print("  ✓ Downloaded")
    _download_splits("caltech-101", ds_root)


def dl_oxford_pets(data_root: Path):
    """oxford_pets/ — torchvision.OxfordIIITPet (renamed from oxford-iiit-pet)"""
    ds_root = data_root / "oxford_pets"
    if (ds_root / "images").exists():
        print("  ✓ Already exists")
    else:
        tvd.OxfordIIITPet(root=str(data_root), download=True)
        src = data_root / "oxford-iiit-pet"
        if src.exists() and not ds_root.exists():
            src.rename(ds_root)
        print("  ✓ Downloaded")
    _download_splits("oxford_pets", ds_root)


def dl_oxford_flowers(data_root: Path):
    """oxford_flowers/ — torchvision.Flowers102 (renamed from flowers-102)"""
    ds_root = data_root / "oxford_flowers"
    if (ds_root / "jpg").exists():
        print("  ✓ Already exists")
    else:
        tvd.Flowers102(root=str(data_root), download=True)
        src = data_root / "flowers-102"
        if src.exists() and not ds_root.exists():
            src.rename(ds_root)
        print("  ✓ Downloaded")
    _download_splits("oxford_flowers", ds_root)


def dl_food101(data_root: Path):
    """food-101/ — torchvision.Food101"""
    ds_root = data_root / "food-101"
    if (ds_root / "images").exists():
        print("  ✓ Already exists")
    else:
        tvd.Food101(root=str(data_root), download=True)
        print("  ✓ Downloaded")
    _download_splits("food-101", ds_root)


def dl_fgvc_aircraft(data_root: Path):
    """fgvc_aircraft/ — torchvision.FGVCAircraft (restructured from fgvc-aircraft-2013b)"""
    ds_root = data_root / "fgvc_aircraft"
    if (ds_root / "images").exists():
        print("  ✓ Already exists")
    else:
        tvd.FGVCAircraft(root=str(data_root), download=True)
        # Restructure: fgvc-aircraft-2013b/data/* → fgvc_aircraft/*
        tv_path = data_root / "fgvc-aircraft-2013b"
        data_dir = tv_path / "data"
        if data_dir.exists():
            ds_root.mkdir(parents=True, exist_ok=True)
            for item in data_dir.iterdir():
                dest = ds_root / item.name
                if not dest.exists():
                    shutil.move(str(item), str(dest))
            shutil.rmtree(str(tv_path), ignore_errors=True)
        print("  ✓ Downloaded")
    _download_splits("fgvc_aircraft", ds_root)


def dl_dtd(data_root: Path):
    """dtd/ — torchvision.DTD"""
    ds_root = data_root / "dtd"
    if (ds_root / "images").exists():
        print("  ✓ Already exists")
    else:
        tvd.DTD(root=str(data_root), download=True)
        print("  ✓ Downloaded")
    _download_splits("dtd", ds_root)


def dl_eurosat(data_root: Path):
    """eurosat/2750/ — torchvision.EuroSAT (with fallback to HuggingFace mirror)"""
    ds_root = data_root / "eurosat"
    if (ds_root / "2750").exists():
        print("  ✓ Already exists")
    else:
        try:
            tvd.EuroSAT(root=str(data_root), download=True)
        except Exception as e:
            print(f"  ⚠ torchvision failed ({e}), trying HuggingFace mirror...")
            from urllib.request import urlretrieve
            ds_root.mkdir(parents=True, exist_ok=True)
            zip_path = ds_root / "EuroSAT.zip"
            urlretrieve(
                "https://huggingface.co/datasets/torchgeo/eurosat/resolve/main/EuroSAT.zip",
                str(zip_path),
            )
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(str(ds_root))
            zip_path.unlink(missing_ok=True)
        # Verify/fix: make sure eurosat/2750/ exists
        if not (ds_root / "2750").exists():
            for p in data_root.rglob("2750"):
                if p.is_dir() and p.parent != ds_root:
                    shutil.move(str(p), str(ds_root / "2750"))
                    break
        print("  ✓ Downloaded")
    _download_splits("eurosat", ds_root)


# ── Manual downloads (not in torchvision) ─────────────────────────────

def dl_stanford_cars(data_root: Path):
    """stanford_cars/ — HuggingFace (tanganke/stanford_cars), no auth needed"""
    ds_root = data_root / "stanford_cars"
    if (ds_root / "cars_train").exists():
        print("  ✓ Already exists")
    else:
        from datasets import load_dataset
        print("  Downloading from HuggingFace (tanganke/stanford_cars)...")
        hf_ds = load_dataset("tanganke/stanford_cars")
        for split_name, folder_name in [("train", "cars_train"), ("test", "cars_test")]:
            split_dir = ds_root / folder_name
            split_dir.mkdir(parents=True, exist_ok=True)
            split_data = hf_ds[split_name]
            for i, example in enumerate(split_data):
                img_path = split_dir / f"{i+1:05d}.jpg"
                if not img_path.exists():
                    img = example["image"]
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                    img.save(str(img_path), "JPEG", quality=95)
            print(f"    {folder_name}: {len(split_data):,} images")
        print("  ✓ Downloaded")
    _download_splits("stanford_cars", ds_root)


def dl_sun397(data_root: Path):
    """sun397/SUN397/ — HuggingFace datasets (torchvision URL is dead)"""
    ds_root = data_root / "sun397"
    sun_dir = ds_root / "SUN397"
    if sun_dir.exists() and any(sun_dir.rglob("*.jpg")):
        print("  ✓ Already exists")
    else:
        from datasets import load_dataset
        print("  Downloading from HuggingFace (1aurent/SUN397) — ~108K images, may take a while...")
        ds = load_dataset("1aurent/SUN397", split="train", trust_remote_code=True)
        label_names = ds.features["label"].names
        per_class = {}
        for i, example in enumerate(ds):
            idx = example["label"]
            cat = label_names[idx].lstrip("/")
            class_dir = sun_dir / cat
            class_dir.mkdir(parents=True, exist_ok=True)
            per_class[idx] = per_class.get(idx, 0) + 1
            img_path = class_dir / f"sun_{idx:03d}_{per_class[idx]:05d}.jpg"
            if not img_path.exists():
                img = example["image"]
                if img.mode != "RGB":
                    img = img.convert("RGB")
                img.save(str(img_path), "JPEG", quality=95)
            if (i + 1) % 10000 == 0:
                print(f"    {i+1:,} / {len(ds):,} images...")
        print(f"  ✓ Saved {len(ds):,} images")
    _download_splits("sun397", ds_root)


def dl_ucf101(data_root: Path):
    """ucf101/UCF-101-midframes/ — Google Drive (midframes not in torchvision)"""
    ds_root = data_root / "ucf101"
    if (ds_root / "UCF-101-midframes").exists():
        print("  ✓ Already exists")
    else:
        ds_root.mkdir(parents=True, exist_ok=True)
        zip_path = ds_root / "UCF-101-midframes.zip"
        print("  Downloading UCF-101 midframes from Google Drive...")
        gdown.download(id="10Jqome3vtUA2keJkNanAiFpgbyC9Hc2O", output=str(zip_path), quiet=False)
        if zip_path.exists():
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(str(ds_root))
            zip_path.unlink(missing_ok=True)
            print("  ✓ Downloaded")
    _download_splits("ucf101", ds_root)


# ── Registry ─────────────────────────────────────────────────────────
ALL_DATASETS = {
    "caltech-101":    dl_caltech101,
    "oxford_pets":    dl_oxford_pets,
    "stanford_cars":  dl_stanford_cars,
    "oxford_flowers": dl_oxford_flowers,
    "food-101":       dl_food101,
    "fgvc_aircraft":  dl_fgvc_aircraft,
    "sun397":         dl_sun397,
    "dtd":            dl_dtd,
    "eurosat":        dl_eurosat,
    "ucf101":         dl_ucf101,
}

## Step 5 — Download All Datasets

Runs through each dataset: download via torchvision (or fallback), restructure folders if needed, then fetch CoOp split JSONs.

In [ ]:
data_root = Path(DATA_ROOT)
data_root.mkdir(parents=True, exist_ok=True)

datasets = SELECTED_DATASETS or list(ALL_DATASETS.keys())

print(f"{'='*60}")
print(f"  CoOp Dataset Downloader  (torchvision + fallbacks)")
print(f"{'='*60}")
print(f"  DATA_ROOT : {data_root}")
print(f"  Datasets  : {', '.join(datasets)}")
print(f"{'='*60}\n")

failed = []
for i, name in enumerate(datasets, 1):
    print(f"\n{'─'*60}")
    print(f"  [{i}/{len(datasets)}] {name}")
    print(f"{'─'*60}")
    try:
        ALL_DATASETS[name](data_root)
        print(f"  ✅ {name} — done")
    except Exception as e:
        print(f"  ❌ {name} — FAILED: {e}")
        import traceback; traceback.print_exc()
        failed.append(name)

succeeded = [n for n in datasets if n not in failed]
print(f"\n{'='*60}")
print(f"  Summary")
print(f"{'='*60}")
print(f"  ✅ Succeeded ({len(succeeded)}): {', '.join(succeeded) or '—'}")
if failed:
    print(f"  ❌ Failed    ({len(failed)}): {', '.join(failed)}")
print(f"{'='*60}")

## Fix: Folder structure of "dtd" dataset

In [ ]:
import shutil
import os

# Set your paths
parent_dtd = "/content/drive/MyDrive/AML Dataset/dtd"
nested_dtd = "/content/drive/MyDrive/AML Dataset/dtd/dtd"

# Move all items from the nested folder to the parent folder
for item in os.listdir(nested_dtd):
    source = os.path.join(nested_dtd, item)
    destination = os.path.join(parent_dtd, item)
    shutil.move(source, destination)

# Remove the empty nested folder
os.rmdir(nested_dtd)
print("Files moved and nested folder deleted successfully!")

## Step 6 — Verify All Datasets

Checks that every expected folder and split JSON exists and counts the files inside.

In [ ]:

EXPECTED = {
    "caltech-101":    {"folders": ["101_ObjectCategories"], "splits": ["split_zhou_Caltech101.json"]},
    "oxford_pets":    {"folders": ["images", "annotations"], "splits": ["split_zhou_OxfordPets.json"]},
    "stanford_cars":  {"folders": ["cars_train", "cars_test"], "splits": ["split_zhou_StanfordCars.json"]},
    "oxford_flowers": {"folders": ["jpg"], "splits": ["split_zhou_OxfordFlowers.json", "cat_to_name.json", "imagelabels.mat"]},
    "food-101":       {"folders": ["images", "meta"], "splits": ["split_zhou_Food101.json"]},
    "fgvc_aircraft":  {"folders": ["images"], "splits": []},
    "sun397":         {"folders": ["SUN397"], "splits": ["split_zhou_SUN397.json"]},
    "dtd":            {"folders": ["images", "imdb", "labels"], "splits": ["split_zhou_DescribableTextures.json"]},
    "eurosat":        {"folders": ["2750"], "splits": ["split_zhou_EuroSAT.json"]},
    "ucf101":         {"folders": ["UCF-101-midframes"], "splits": ["split_zhou_UCF101.json"]},
}

data_root = Path(DATA_ROOT)
datasets = SELECTED_DATASETS or list(ALL_DATASETS.keys())

print(f"{'='*60}")
print(f"  Post-Download Verification")
print(f"{'='*60}\n")

all_good = True
for name in datasets:
    ds_root = data_root / name
    exp = EXPECTED[name]
    issues = []

    # Check expected folders
    for folder in exp["folders"]:
        p = ds_root / folder
        if p.exists() and p.is_dir():
            n_files = sum(1 for _ in p.rglob("*") if _.is_file())
            print(f"  ✅ {name}/{folder}/  ({n_files:,} files)")
            if n_files == 0:
                issues.append(f"{folder}/ is empty")
        else:
            print(f"  ❌ {name}/{folder}/  — MISSING")
            issues.append(f"{folder}/ not found")

    # Check split JSONs / extra files
    for fname in exp["splits"]:
        p = ds_root / fname
        if p.exists():
            size_kb = p.stat().st_size / 1024
            print(f"  ✅ {name}/{fname}  ({size_kb:.1f} KB)")
        else:
            print(f"  ❌ {name}/{fname}  — MISSING")
            issues.append(f"{fname} not found")

    if issues:
        all_good = False
        print(f"  ⚠️  {name}: {len(issues)} issue(s)")
    print()

print(f"{'='*60}")
if all_good:
    print(f"  ✅ All datasets verified — ready to use with CoOp/Dassl")
else:
    print(f"  ⚠️  Some datasets have issues — check output above")
print(f"{'='*60}")